# Ask-SLM Runner
Minimal notebook flow: load metadata, prepare mappings, build analysis-wide dataset, and launch managers.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from Accessors.SraAccessor.sra_metadata_accessor import SRAMetadataAccessor
from Accessors.SraAccessor.sra_taxonomy_accessor import SRATaxonomyAccessor
from processors.sra_processor import SRAProcessor

def require_vars(names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError(f"Missing required variables: {', '.join(missing)}")

In [2]:
view_name = "PhagesInBlood"
input_dir = Path("Inputs") / view_name
metadata_source = input_dir / "metadata.csv"
taxonomy_source = input_dir / "taxonomy.csv"

In [ ]:
metadata_table = SRAMetadataAccessor.get(metadata_source)
display(metadata_table.head())

,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
0,SRR16574708,OTHER,GEO,public,SRX12776662,GSM5655983,HiSeq X Ten,PAIRED,other,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[s3, ncbi, gs]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655983}, {k=bases...","{""geo_accession_exp"": [""GSM5655983""], ""bases"":...",1,PRJNA774626
1,SRR16574682,OTHER,GEO,public,SRX12776636,GSM5655957,HiSeq X Ten,PAIRED,other,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=geo_accession_exp, v=GSM5655957}, {k=bases...","{""geo_accession_exp"": [""GSM5655957""], ""bases"":...",1,PRJNA774626
2,SRR16574640,OTHER,GEO,public,SRX12776594,GSM5655915,HiSeq X Ten,PAIRED,other,GENOMIC,...,NaN,NaN,NaN,"[run.zq, sra, fastq]","[s3, gs, ncbi]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655915}, {k=bases...","{""geo_accession_exp"": [""GSM5655915""], ""bases"":...",1,PRJNA774626
3,SRR16574660,OTHER,GEO,public,SRX12776614,GSM5655935,HiSeq X Ten,PAIRED,other,GENOMIC,...,NaN,NaN,NaN,"[fastq, sra, run.zq]","[gs, s3, ncbi]","[gs.us-east1, s3.us-east-1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655935}, {k=bases...","{""geo_accession_exp"": [""GSM5655935""], ""bases"":...",1,PRJNA774626
4,SRR16574696,OTHER,GEO,public,SRX12776650,GSM5655971,HiSeq X Ten,PAIRED,other,GENOMIC,...,NaN,NaN,NaN,"[sra, run.zq, fastq]","[ncbi, s3, gs]","[s3.us-east-1, gs.us-east1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655971}, {k=bases...","{""geo_accession_exp"": [""GSM5655971""], ""bases"":...",1,PRJNA774626


In [ ]:
sra_processor = SRAProcessor()
config_dir = Path("mapping_configs")
project_ids = sorted(metadata_table["bioproject"].dropna().unique())

def refresh_templates(metadata_df, project_ids, processor, config_dir):
    rows = []
    for project_id in project_ids:
        project_df = metadata_df[metadata_df["bioproject"] == project_id].copy()
        processor.refresh_template(project_df, project_id)
        config_path = config_dir / f"{project_id}.json"
        rows.append(
            {
                "project_id": project_id,
                "config_exists": config_path.exists(),
                "config_path": str(config_path),
            }
        )
    return pd.DataFrame(rows)

project_status_df = refresh_templates(
    metadata_df=metadata_table,
    project_ids=project_ids,
    processor=sra_processor,
    config_dir=config_dir,
)
display(project_status_df)

,project_id,config_exists,config_path
0,PRJEB50557,True,mapping_configs/PRJEB50557.json
1,PRJEB64933,True,mapping_configs/PRJEB64933.json
2,PRJEB89352,True,mapping_configs/PRJEB89352.json
3,PRJNA774626,True,mapping_configs/PRJNA774626.json


In [5]:
from managers.MappingConfigManager.mapping_config_manager import MappingConfigManager

mapping_manager = MappingConfigManager()
mapping_manager.run()

✅ Loaded 27 fields from Metalog schema
📋 Loaded 4 project(s)
📝 Found 5 existing target field(s): ['Group', 'environment_material', 'sex', 'subject_id', 'timepoint']



In [ ]:
# After mappings configured and saved in <project_id>.json files, we now use it to enrich the metadata.
def enrich_metadata(metadata_table, project_ids, processor, config_dir):
    processed_parts = []
    missing_configs = []

    for project_id in project_ids:
        project_config_path = config_dir / f"{project_id}.json"
        if not project_config_path.exists():
            missing_configs.append(project_id)
            continue

        project_metadata = metadata_table[metadata_table["bioproject"] == project_id].copy()
        processed_parts.append(processor.process(project_metadata, project_id))

    if missing_configs:
        missing = ", ".join(missing_configs)
        raise FileNotFoundError(f"Missing mapping config for: {missing}")

    if not processed_parts:
        raise RuntimeError("No projects were processed")

    enriched = pd.concat(processed_parts, ignore_index=True)
    mapped = [c for c in enriched.columns if c not in metadata_table.columns]
    return enriched, mapped

enriched_metadata, mapped_cols = enrich_metadata(
    metadata_table=metadata_table,
    project_ids=project_ids,
    processor=sra_processor,
    config_dir=config_dir,
)

summary_df = pd.DataFrame(
    {
        "metric": ["samples", "projects", "mapped_columns"],
        "value": [len(enriched_metadata), enriched_metadata["bioproject"].nunique(), len(mapped_cols)],
    }
)
display(summary_df)
display(enriched_metadata.head())

✅ Found configuration for project PRJEB50557. Applying mappings...

🚫 Project excluded by filter configuration

⚠️ All samples were filtered out

✅ Found configuration for project PRJEB64933. Applying mappings...

🚫 Project excluded by filter configuration

...

🚫 Project excluded by filter configuration

⚠️ All samples were filtered out

✅ Found configuration for project PRJNA774626. Applying mappings...

,metric,value
0,samples,86
1,projects,1
2,mapped_columns,3


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject,environment_material,sex,Group
0,SRR16574708,OTHER,GEO,public,SRX12776662,GSM5655983,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[s3, ncbi, gs]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655983}, {k=bases...","{""geo_accession_exp"": [""GSM5655983""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer
1,SRR16574682,OTHER,GEO,public,SRX12776636,GSM5655957,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=geo_accession_exp, v=GSM5655957}, {k=bases...","{""geo_accession_exp"": [""GSM5655957""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
2,SRR16574640,OTHER,GEO,public,SRX12776594,GSM5655915,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[run.zq, sra, fastq]","[s3, gs, ncbi]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655915}, {k=bases...","{""geo_accession_exp"": [""GSM5655915""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
3,SRR16574660,OTHER,GEO,public,SRX12776614,GSM5655935,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, sra, run.zq]","[gs, s3, ncbi]","[gs.us-east1, s3.us-east-1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655935}, {k=bases...","{""geo_accession_exp"": [""GSM5655935""], ""bases"":...",1,PRJNA774626,plasma,female,healthy
4,SRR16574696,OTHER,GEO,public,SRX12776650,GSM5655971,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[sra, run.zq, fastq]","[ncbi, s3, gs]","[s3.us-east-1, gs.us-east1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655971}, {k=bases...","{""geo_accession_exp"": [""GSM5655971""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer


In [ ]:
TAXON_KEY_SEP = "||"

def build_canonical_analysis_dataset(enriched_metadata_df, taxonomy_df, taxon_key_sep=TAXON_KEY_SEP):
    metadata_columns = list(enriched_metadata_df.columns)
    lineage_columns = [c for c in ["ileft", "iright"] if c in taxonomy_df.columns]

    taxonomy_for_wide = taxonomy_df[["acc", "tax_id", "rank", "name", "total_count"] + lineage_columns].copy()
    taxonomy_for_wide["taxon_key"] = (
        taxonomy_for_wide["rank"].astype(str)
        + taxon_key_sep
        + taxonomy_for_wide["name"].astype(str)
        + taxon_key_sep
        + taxonomy_for_wide["tax_id"].astype(str)
    )
    taxonomy_for_wide = (
        taxonomy_for_wide[["acc", "taxon_key", "total_count"]]
        .groupby(["acc", "taxon_key"], as_index=False)["total_count"]
        .sum()
    )

    taxa_counts_wide = taxonomy_for_wide.pivot(
        index="acc",
        columns="taxon_key",
        values="total_count",
    ).fillna(0)

    lookup_base = taxonomy_df[["tax_id", "rank", "name"] + lineage_columns].copy()
    lookup_base["taxon_key"] = (
        lookup_base["rank"].astype(str)
        + taxon_key_sep
        + lookup_base["name"].astype(str)
        + taxon_key_sep
        + lookup_base["tax_id"].astype(str)
    )
    taxon_lookup_columns = ["taxon_key", "rank", "name", "tax_id"] + lineage_columns
    taxon_lookup_df = lookup_base[taxon_lookup_columns].drop_duplicates(subset=["taxon_key"])

    abundance_metadata_matrix = enriched_metadata_df.merge(taxa_counts_wide.reset_index(), on="acc", how="left")
    taxa_feature_columns = [c for c in abundance_metadata_matrix.columns if c not in metadata_columns]
    abundance_metadata_matrix[taxa_feature_columns] = abundance_metadata_matrix[taxa_feature_columns].fillna(0)
    abundance_matrix = abundance_metadata_matrix[["acc"] + taxa_feature_columns].set_index("acc")

    return abundance_metadata_matrix, taxon_lookup_df, metadata_columns, abundance_matrix

taxonomy_table = SRATaxonomyAccessor.get(taxonomy_source)
abundance_metadata_matrix, taxon_lookup_df, metadata_columns, abundance_matrix = build_canonical_analysis_dataset(
    enriched_metadata_df=enriched_metadata,
    taxonomy_df=taxonomy_table,
    taxon_key_sep=TAXON_KEY_SEP,
)

output_dir = Path("results") / view_name
output_dir.mkdir(parents=True, exist_ok=True)
taxa_matrix_csv_path = output_dir / "Ask_SLM_Taxon_Abundance_Matrix.csv"
canonical_dataset_csv_path = output_dir / "Ask_SLM_Analysis_Wide_Dataset.csv"
abundance_matrix.to_csv(taxa_matrix_csv_path)
abundance_metadata_matrix.to_csv(canonical_dataset_csv_path, index=False)

display(
    pd.DataFrame(
        {
            "artifact": ["abundance_matrix", "abundance_metadata_matrix"],
            "path": [str(taxa_matrix_csv_path), str(canonical_dataset_csv_path)],
        }
    )
)
display(abundance_metadata_matrix[metadata_columns].head())

,artifact,path
0,feature_matrix,results/PhagesInBlood/Ask_SLM_Taxon_Abundance_...
1,analysis_wide,results/PhagesInBlood/Ask_SLM_Analysis_Wide_Da...


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject,environment_material,sex,Group
0,SRR16574708,OTHER,GEO,public,SRX12776662,GSM5655983,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[s3, ncbi, gs]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655983}, {k=bases...","{""geo_accession_exp"": [""GSM5655983""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer
1,SRR16574682,OTHER,GEO,public,SRX12776636,GSM5655957,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=geo_accession_exp, v=GSM5655957}, {k=bases...","{""geo_accession_exp"": [""GSM5655957""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
2,SRR16574640,OTHER,GEO,public,SRX12776594,GSM5655915,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[run.zq, sra, fastq]","[s3, gs, ncbi]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655915}, {k=bases...","{""geo_accession_exp"": [""GSM5655915""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
3,SRR16574660,OTHER,GEO,public,SRX12776614,GSM5655935,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, sra, run.zq]","[gs, s3, ncbi]","[gs.us-east1, s3.us-east-1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655935}, {k=bases...","{""geo_accession_exp"": [""GSM5655935""], ""bases"":...",1,PRJNA774626,plasma,female,healthy
4,SRR16574696,OTHER,GEO,public,SRX12776650,GSM5655971,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[sra, run.zq, fastq]","[ncbi, s3, gs]","[s3.us-east-1, gs.us-east1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655971}, {k=bases...","{""geo_accession_exp"": [""GSM5655971""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer


In [ ]:
from managers.AlphaDiversityManager.alpha_diversity_manager import AlphaDiversityManager

require_vars(["abundance_metadata_matrix", "taxon_lookup_df", "metadata_columns"])

alpha_manager = AlphaDiversityManager(
    analysis_wide_df=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
 )
alpha_manager.run()

📊 Building taxonomy lineage database...
Building lineage database from 4233 rows...
✅ Built lineage database with 4233 unique taxa
📋 Standard ranks available: ['superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
   (+ 25 additional ranks available)


In [ ]:
from managers.TaxonomicAbundanceManager import TaxonomicAbundanceManager

require_vars(["abundance_metadata_matrix", "taxon_lookup_df", "metadata_columns"])

abundance_manager = TaxonomicAbundanceManager(
    abundance_metadata_matrix=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
 )
abundance_manager.run()

In [ ]:
from managers.LongitudinalAnalysisManager import LongitudinalAnalysisManager

require_vars(["enriched_df", "sra_metadata_df"])

longitudinal_manager = LongitudinalAnalysisManager(enriched_metadata, set(metadata_table.columns))
longitudinal_manager.run()

HTML(value='\n            <div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); \n       …